<a href="https://colab.research.google.com/github/Mashael-fs/IT326-Cardiovascular-Disease/blob/main/Reports%20Folder/Phase3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 3

## **Classification**


### importing library

In [ ]:
# Import all necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn import metrics
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from tabulate import tabulate
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import matplotlib.cm as cm
from sklearn.metrics import silhouette_score, silhouette_samples



This cell imports all required libraries for data manipulation, visualization, machine learning (Decision Tree), and evaluation metrics.

### Loading the Dataset

In [ ]:
# load dataset
df = pd.read_csv("Preprocessed_dataset.csv")

print("Dataset shape:", df.shape)
print(df.head())
print("\nColumns:", df.columns.tolist())

This cell loads the preprocessed cardiovascular disease dataset and displays its basic information.

### Selecting the Features and Target

In [ ]:
# we should drop id, cardio, age_years, and Age_Group columns before training process
X = df.drop(
    columns=["id", "cardio", "age_years", "Age_Group"],
    errors="ignore"
)

y = df["cardio"] # Target variable

print("Feature columns:", X.columns.tolist())
print("\nClass distribution:")
print(y.value_counts().sort_index())

This cell separates the input features from the target class. The cardio column is used as the target variable, where 0 represents the absence of cardiovascular disease and 1 represents its presence.

## Gini Index – Decision Tree Classification Using Three Data Partitions

In [ ]:
# Gini Decision Tree - Three Data Partitions

partitions = [0.1, 0.2, 0.3]
gini_results = []

for test_size in partitions:

    train_percent = round((1 - test_size) * 100)
    test_percent = round(test_size * 100)

    print(f"\nGini - {train_percent}% Training / {test_percent}% Testing")

    # Split dataset into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=1
    )

    # Create Decision Tree classifier using Gini Index
    clf_gini = DecisionTreeClassifier(
        criterion="gini", random_state=1
    )

    # Train Decision Tree classifier
    clf_gini.fit(X_train, y_train)

    # Predict the test dataset
    y_pred_gini = clf_gini.predict(X_test)

    # Compute and print confusion matrix
    cm = confusion_matrix(y_test, y_pred_gini)
    print("Confusion Matrix:")
    print(cm)
    tn, fp, fn, tp = cm.ravel()

    # Display confusion matrix
    ConfusionMatrixDisplay.from_estimator(
        clf_gini, X_test, y_test, display_labels=["No cardio", "cardio"]
    )
    plt.title(f"Gini - {train_percent}% Training / {test_percent}% Testing")
    plt.show()

    # Calculate accuracy, error rate, sensitivity, specificity, precision
    accuracy = metrics.accuracy_score(y_test, y_pred_gini)
    error_rate = 1 - accuracy
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0

    print(f"Accuracy:    {accuracy * 100:.2f} %")
    print(f"Error rate:  {error_rate * 100:.2f} %")
    print(f"Sensitivity(recall): {sensitivity * 100:.2f} %")
    print(f"Specificity: {specificity * 100:.2f} %")
    print(f"Precision:   {precision * 100:.2f} %\n")

    # Save results for comparison
    gini_results.append({
    'Test Size': f"{train_percent}% Training / {test_percent}% Testing",
    'Accuracy': accuracy,
    'Error Rate': error_rate,
    'Sensitivity(recall)': sensitivity,
    'Specificity': specificity,
    'Precision': precision
    })

# Comparison of Gini Results
print("\nComparison of Gini Results")

headers = ["Data Partition", "Accuracy"]

data = [
    [result['Test Size'], f"{result['Accuracy'] * 100:.2f}%"]
    for result in gini_results
]

print(tabulate(data, headers=headers, tablefmt="fancy_grid"))

The Decision Tree classifier was evaluated using the Gini Index across three training/testing partitions: 90/10, 80/20, and 70/30. Since the classification procedure remains identical for all three experiments and only the training/testing proportions change, a for loop was used to avoid repeating the same code. In each iteration, the dataset was split, a new Decision Tree was trained using the Gini Index, and its performance was evaluated using Accuracy, Error rate, Sensitivity(recall), Specificity, Precision and a confusion matrix. The results were then stored in a list to generate a comparison table.

Results:

90/10 partition: Achieved an accuracy of 62.04%, using the largest training set and smallest testing set.
80/20 partition: Achieved an accuracy of 62.76%, representing an increase of 0.72 percentage points compared with 90/10.
70/30 partition: Achieved the highest accuracy of 63.34%, outperforming 90/10 by 1.30 percentage points and 80/20 by 0.58 percentage points.

Findings:

The 70/30 partition produced the highest classification accuracy among the three experiments, despite using the smallest training set. Unlike the Entropy results, the Gini Index showed a consistent increase in accuracy as the training proportion decreased from 90% to 70%. However, the differences were relatively small, with a maximum difference of 1.30 percentage points across the tested partitions.

Based on the observed results, 70% training and 30% testing was the best-performing partition for the Gini Decision Tree. However, this conclusion is limited to the evaluated splits and does not establish that 70/30 will always perform best.

## Entropy (Information Gain) – Decision Tree Classification Using Three Data Partitions

In [ ]:
# Entropy Decision Tree - Three Data Partitions

partitions = [0.1, 0.2, 0.3]
entropy_results = []

for test_size in partitions:

    train_percent = round((1 - test_size) * 100)
    test_percent = round(test_size * 100)

    print(f"\nEntropy - {train_percent}% Training / {test_percent}% Testing")

    # Split dataset into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=1
    )

    # Create Decision Tree classifier using Entropy
    clf_entropy = DecisionTreeClassifier(
        criterion="entropy", random_state=1
    )

    # Train Decision Tree classifier
    clf_entropy.fit(X_train, y_train)

    # Predict the test dataset
    y_pred_entropy = clf_entropy.predict(X_test)

    # Compute and print confusion matrix
    cm = confusion_matrix(y_test, y_pred_entropy)
    print("Confusion Matrix:")
    print(cm)
    tn, fp, fn, tp = cm.ravel()

    # Display confusion matrix
    ConfusionMatrixDisplay.from_estimator(
        clf_entropy, X_test, y_test, display_labels=["No cardio", "cardio"]
    )
    plt.title(f"Entropy - {train_percent}% Training / {test_percent}% Testing")
    plt.show()

    # Calculate accuracy, error rate, sensitivity, specificity, precision
    accuracy = metrics.accuracy_score(y_test, y_pred_entropy)
    error_rate = 1 - accuracy
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0

    print(f"Accuracy:    {accuracy * 100:.2f} %")
    print(f"Error rate:  {error_rate * 100:.2f} %")
    print(f"Sensitivity(recall): {sensitivity * 100:.2f} %")
    print(f"Specificity: {specificity * 100:.2f} %")
    print(f"Precision:   {precision * 100:.2f} %\n")

    # Save results for comparison
    entropy_results.append({
    'Test Size': f"{train_percent}% Training / {test_percent}% Testing",
    'Accuracy': accuracy,
    'Error Rate': error_rate,
    'Sensitivity(recall)': sensitivity,
    'Specificity': specificity,
    'Precision': precision
    })

# Comparison of Entropy Results
print("\nComparison of Entropy Results")

headers = ["Data Partition", "Accuracy"]

data = [
    [result['Test Size'], f"{result['Accuracy'] * 100:.2f}%"]
    for result in entropy_results
]

print(tabulate(data, headers=headers, tablefmt="fancy_grid"))

The Decision Tree classifier was evaluated using Entropy (Information Gain) across three training/testing partitions: 90/10, 80/20, and 70/30. Since the classification procedure remains identical for all three experiments and only the training/testing proportions change, a for loop was used to avoid repeating the same code. In each iteration, the dataset was split, a new Decision Tree was trained using Entropy, and its performance was evaluated using Accuracy, Error rate, Sensitivity(recall), Specificity, Precision and a confusion matrix. The results were then stored in a list to generate a comparison table.

Results:

90/10 partition: Achieved an accuracy of 63.11%, using the largest training set and smallest testing set. 80/20 partition: Achieved the same accuracy of 63.11%, showing no change compared with the 90/10 partition. 70/30 partition: Achieved the highest accuracy of 63.77%, outperforming both 90/10 and 80/20 by 0.66 percentage points.

Findings:

The 70/30 partition produced the highest classification accuracy among the three experiments, despite using the smallest training set. Unlike the Gini Index results, the Entropy accuracy did not consistently increase as the training proportion decreased, since the 90/10 and 80/20 partitions produced identical accuracy values. However, the differences were relatively small, with a maximum difference of 0.66 percentage points across the tested partitions.

Based on the observed results, 70% training and 30% testing was the best-performing partition for the Entropy Decision Tree. However, this conclusion is limited to the evaluated splits and does not establish that 70/30 will always perform best.

## Classification Results Comparison – Gini Index vs. Entropy (Information Gain)

In [ ]:

# Convert Gini and Entropy results into DataFrames
results_gini = pd.DataFrame(
    gini_results, columns=["Test Size", "Accuracy"]
)
results_gini.rename(columns={"Test Size": "Data Partition"}, inplace=True)
results_gini["Measure"] = "Gini"

results_entropy = pd.DataFrame(
    entropy_results, columns=["Test Size", "Accuracy"]
)
results_entropy.rename(columns={"Test Size": "Data Partition"}, inplace=True)
results_entropy["Measure"] = "Entropy"

# Combine results into one DataFrame
comparison_df = pd.concat(
    [results_gini, results_entropy],
    ignore_index=True
)

# Display all comparison results
print("\nCOMPARISON RESULTS")
print(comparison_df.to_string(index=False))

# Create an accuracy comparison table
pivot_acc = comparison_df.pivot(
    index="Data Partition",
    columns="Measure",
    values="Accuracy"
)

print("\nAccuracy Comparison (%):")
print((pivot_acc * 100).round(2))


The classification results obtained using the Gini Index and Entropy (Information Gain) were combined into one DataFrame to allow direct comparison across the three training/testing partitions. The results for each measure were first converted into separate DataFrames and labeled according to the attribute selection measure used. They were then combined using pd.concat(). A pivot table was also created to display the accuracy of Gini and Entropy side by side for each partition.

Results:

For the 90/10 partition, Gini achieved an accuracy of 62.04%, while Entropy achieved 63.11%, giving Entropy an advantage of 1.07 percentage points. For the 80/20 partition, Gini achieved 62.76%, compared with 63.11% for Entropy, a difference of 0.35 percentage points. For the 70/30 partition, Gini achieved 63.34%, while Entropy achieved 63.77%, with Entropy outperforming Gini by 0.43 percentage points.

Findings:

Entropy achieved a higher classification accuracy than Gini across all three tested partitions. Both attribute selection measures obtained their highest accuracy using the 70% training / 30% testing partition.

Although Entropy consistently outperformed Gini, the differences between the two measures were relatively small. Based on the observed results, Entropy was the better-performing attribute selection measure for this dataset, while the 70/30 split was the best-performing partition for both Decision Tree models.

## Decision Tree Visualization – Gini Index and Entropy (70/30 Partition)

In [ ]:
# Split dataset using 70% training and 30% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=1
)

# Train the Gini Decision Tree
tree_gini = DecisionTreeClassifier(
    criterion="gini",
    random_state=1
)
tree_gini.fit(X_train, y_train)

# Train the Entropy Decision Tree
tree_entropy = DecisionTreeClassifier(
    criterion="entropy",
    random_state=1
)
tree_entropy.fit(X_train, y_train)

# Visualize the Gini Decision Tree
plt.figure(figsize=(18, 8))
plot_tree(
    tree_gini,
    feature_names=X.columns,
    class_names=["No cardio", "cardio"],
    filled=True,
    rounded=True,
    max_depth=2,
    fontsize=10
)
plt.title("Gini Decision Tree – 70% Training / 30% Testing")
plt.show()

# Visualize the Entropy Decision Tree
plt.figure(figsize=(18, 8))
plot_tree(
    tree_entropy,
    feature_names=X.columns,
    class_names=["No cardio", "cardio"],
    filled=True,
    rounded=True,
    max_depth=2,
    fontsize=10
)
plt.title("Entropy Decision Tree – 70% Training / 30% Testing")
plt.show()


The Gini Index and Entropy Decision Trees were visualized using the 70% training / 30% testing partition, since this partition achieved the highest classification accuracy for both measures. The Gini model achieved an accuracy of 63.34%, while the Entropy model achieved 63.77%. To make the trees easier to interpret, only the first three levels were displayed using max_depth=2; the models themselves were not restricted to this depth.


Results:

Both Decision Trees selected systolic blood pressure (ap_hi) as the root attribute, using the condition ap_hi <= 0.216. Records satisfying this condition follow the left branch, while records with values greater than 0.216 follow the right branch. The left branch is mainly classified as No cardio, whereas the right branch is mainly classified as cardio.

At the next level, both trees use age on the left side with the condition age <= -0.013, while the right side uses ap_hi <= 0.845. At the following level, cholesterol is used for further splitting on several branches, while ap_hi is used again on the far-right branch. This shows that systolic blood pressure, age, and cholesterol are important attributes in the visible part of both Decision Trees.

Each node shows the splitting condition, impurity value, number of samples, class distribution, and predicted class. The Gini tree displays gini impurity values, while the Entropy tree displays entropy values. The orange nodes mainly represent No cardio, while the blue nodes mainly represent cardio. Darker colors indicate that a larger proportion of the samples in that node belong to the predicted class. The grey boxes containing (...) indicate that additional branches exist but are hidden because the visualization was limited to max_depth=2.


Findings:

The visible structures of the Gini and Entropy trees are very similar, with both selecting the same root attribute and the same main splitting attributes within the displayed levels. However, the impurity calculations differ because the two models use different attribute selection measures.

Based on the classification results, Entropy performed slightly better than Gini on the 70/30 partition, achieving 63.77% compared with 63.34%. Therefore, although both models produced similar visible tree structures, Entropy provided the better classification performance for this dataset.

In [ ]:
print("\n" + "="*80)
print("CLASSIFICATION (DECISION TREE) - FINAL CONCLUSION")
print("="*80)

print("\nOBJECTIVE:")
print("-"*50)
print("""
To predict the cardio target using Decision Tree classifiers with:
• Two attribute selection measures: Gini Index and Entropy (Information Gain)
• Three different training/testing partitions: 90/10, 80/20, and 70/30
• Comparison and evaluation using classification accuracy and confusion matrices
""")

print("\nKEY FINDINGS:")
print("-"*50)
print("""
1. GINI INDEX VS ENTROPY:
   • Entropy achieved higher accuracy than Gini in all three tested partitions
   • The performance difference between the two measures was relatively small
   • Entropy produced the best overall classification result

2. EFFECT OF TRAIN/TEST PARTITION SIZE:
   • For Gini, accuracy increased as the training proportion decreased from 90% to 70%
   • For Entropy, the 90/10 and 80/20 partitions produced almost identical accuracy
   • The 70/30 partition achieved the highest accuracy for both measures

3. BEST CLASSIFICATION RESULTS:
   • Best Gini result: 63.34% accuracy using the 70/30 partition
   • Best Entropy result: 63.77% accuracy using the 70/30 partition
   • Best overall model: Entropy with the 70/30 partition

4. DECISION TREE STRUCTURE:
   • Both models selected ap_hi as the root attribute
   • Age, cholesterol, and ap_hi were also used in the visible tree levels
   • The visible Gini and Entropy tree structures were very similar
""")

print("\nPERFORMANCE SUMMARY:")
print("-"*50)

gini_avg_acc = results_gini["Accuracy"].mean()
entropy_avg_acc = results_entropy["Accuracy"].mean()

print(f"• Average Accuracy (Gini): {gini_avg_acc*100:.2f}%")
print(f"• Average Accuracy (Entropy): {entropy_avg_acc*100:.2f}%")
print("• Best Model: Entropy with 70% Training / 30% Testing (Accuracy: 63.77%)")

print("\nLIMITATIONS:")
print("-"*50)
print("""
• The results are based on the three selected training/testing partitions
• Only two Decision Tree attribute selection measures were compared
• The classification performance depends on the characteristics of the current dataset
""")

print("\nCONCLUSION:")
print("-"*50)
print("""
The Decision Tree classification experiments showed that Entropy (Information Gain)
performed slightly better than the Gini Index for all three evaluated data partitions.

Both measures achieved their highest accuracy using the 70% training / 30% testing
partition. The best overall classification result was obtained using Entropy,
with an accuracy of 63.77%, compared with 63.34% for Gini.

Therefore, based on the evaluated experiments, the Entropy Decision Tree with the
70/30 partition was selected as the best-performing classification configuration
for predicting the cardio target in this dataset.
""")

## **Clustrering**




### K-Means Preprocessing
1.Prepare the data

In [ ]:
# Columns to exclude from clustering
columns_to_drop = ["target", "id","age","cardio"]

# Drop excluded columns and ignore names that are not present
X = df.drop(columns=columns_to_drop, errors="ignore")

# Keep numeric columns only
X = X.select_dtypes(include="number").copy()

# Remove rows with missing values
X = X.dropna()

print("Data shape:", X.shape)
print("Features used:", X.columns.tolist())

2.Scale the features

In [ ]:
# Standardize features so larger-scale values do not dominate the distances
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled data shape:", X_scaled.shape)

K-Means Preprocessing Explanation
Before applying K-Means clustering, the dataset was prepared by excluding the target variable (cardio) and other specified attributes.
Only numerical features were retained, and rows containing missing values were removed. The resulting dataset contained 62,505 records and 11 features.
The selected features were then standardized using StandardScaler() to prevent attributes with larger numerical scales from dominating the Euclidean distance calculations. After standardization, the dataset retained the same dimensions (62,505 × 11).

3.Plot Elbow and Silhouette results

In [ ]:
# Calculate WCSS for K values from 1 to 10
wcss = []
k_range = range(1, 11)

for k in k_range:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

# Plot the Elbow curve
plt.figure(figsize=(10, 6))
plt.plot(k_range, wcss, "bo-", linewidth=2)
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Within-Cluster Sum of Squares (WCSS)")
plt.title("Elbow Method for K Selection")
plt.xticks(list(k_range))
plt.grid(True, alpha=0.3)
plt.show()

# Print the WCSS values
print("WCSS values:")
for k, value in zip(k_range, wcss):
    print(f"K={k}: WCSS={value:.2f}")

Elbow Method – Results and Analysis

The plot shows the WCSS values for different numbers of clusters, from K = 1 to K = 10. The WCSS decreases as the number of clusters increases because data points become closer to their assigned cluster centroids.

The Elbow Method was used to examine the reduction in WCSS and identify suitable values for K. Based on the analysis, K = 6, K = 7, and K = 8 were selected for further evaluation using the WCSS and Silhouette Score to determine the most suitable number of clusters.

4.Compare different K values

In [ ]:
# Compare three K values selected for evaluation
K_values = [6, 7, 8]

kmeans_models = {}
cluster_labels = {}
comparison_results = []

print("K-MEANS CLUSTERING WITH DIFFERENT K VALUES")
print("=" * 50)

for k in K_values:
    print(f"\nTraining K-Means with K={k}")

    # Train the K-Means model and assign cluster labels
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    labels = kmeans.fit_predict(X_scaled)

    # Save the model and labels for later use
    kmeans_models[k] = kmeans
    cluster_labels[k] = labels

    # Calculate evaluation metrics
    wcss = kmeans.inertia_
    sil_score = silhouette_score(X_scaled, labels)

    # Save the results for comparison
    comparison_results.append({
        "K": k,
        "WCSS": wcss,
        "Silhouette": sil_score
    })

    print(f"WCSS (Inertia): {wcss:.2f}")
    print(f"Silhouette Score: {sil_score:.3f}")

# Display a comparison table
comparison_df = pd.DataFrame(comparison_results)
print("\nComparison of K values:")
print(comparison_df.round(3).to_string(index=False))

K-MEANS CLUSTERING WITH DIFFERENT K VALUES

Training K-Means with K=6
WCSS (Inertia): 391903.96
Silhouette Score: 0.193

Training K-Means with K=7
WCSS (Inertia): 360419.98
Silhouette Score: 0.201

Training K-Means with K=8


K-Means Comparison – Results and Analysis

K-Means clustering was evaluated using three different numbers of clusters: K = 6, K = 7, and K = 8. The performance of each configuration was compared using the Within-Cluster Sum of Squares (WCSS) and the Silhouette Score.

The WCSS decreased from 391,903.965 at K = 6 to 360,419.980 at K = 7 and 344,744.000 at K = 8. This indicates that increasing the number of clusters reduces the distances between data points and their assigned centroids.

However, the Silhouette Score provides additional information about cluster cohesion and separation. K = 7 and K = 8 achieved the highest Silhouette Score of 0.201, compared with 0.193 for K = 6.

Although K = 8 produced the lowest WCSS, K = 7 achieved the same highest Silhouette Score with fewer clusters. Therefore, K = 8 was selected as the best-performing configuration because it provides a good balance between clustering quality and model simplicity.

5.Select the final K and run K-Means

In [ ]:
# Set the K value selected from the Elbow and Silhouette results
best_k = 8

# Fit the final K-Means model
final_kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

# Assign each record to a cluster
cluster_labels = final_kmeans.fit_predict(X_scaled)

print("Selected K:", best_k)

Final K-Means Model - Results and Analysis
Based on the Elbow Method and Silhouette Score comparison, K = 8 was selected as the most suitable number of clusters among the
evaluated values. The final K-Means model was trained using the standardized dataset, with random_state=42 and n_init=10. Each record was assigned to one of the three clusters based on its distance from the cluster centroids.

6.Visualize the clusters using PCA

In [ ]:

# Reduce the scaled features to two dimensions for visualization
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_scaled)

# Create three plots for K = 6, 7, 8
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for i, k in enumerate(K_values):

    # Get cluster labels for each K value from the saved models
    labels = kmeans_models[k].labels_

    # Plot the records colored by their cluster
    scatter = axes[i].scatter(
        X_2d[:, 0],
        X_2d[:, 1],
        c=labels,
        cmap="viridis",
        alpha=0.7
    )

    axes[i].set_xlabel("Principal Component 1")
    axes[i].set_ylabel("Principal Component 2")
    axes[i].set_title(f"K-Means Clusters (K={k})")
    axes[i].grid(True, alpha=0.3)

    fig.colorbar(scatter, ax=axes[i], label="Cluster")

plt.tight_layout()
plt.show()


PCA Visualization – Results and Analysis

PCA was applied to reduce the standardized dataset to two dimensions for visualization. The resulting scatter plots compare the clustering results for three different values of K: K = 6, K = 7, and K = 8. Each color represents a different cluster.

The visualization shows that the data points are distributed across the two-dimensional space, with noticeable overlap between clusters in all three configurations. As the number of clusters increases, the data is divided into more groups, resulting in different cluster assignments.

For K = 6, the data is divided into six clusters. For K = 7, it is divided into seven clusters, while K = 8 produces eight clusters. However, the plots show that the clusters are not completely separated, as several groups overlap in the PCA projection.

Overall, PCA provides a visual comparison of the clustering results and helps illustrate how changing the number of clusters affects the distribution of data points.

## Silhouette Score Comparison & Analysis

In [ ]:

import matplotlib.cm as cm

# 1. Create Silhouette plots for K = 6, 7, 8
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Subsample 10,000 records for faster rendering
np.random.seed(42)
subsample_size = min(10000, X_scaled.shape[0])
sub_idx = np.random.choice(
    X_scaled.shape[0],
    size=subsample_size,
    replace=False
)
X_sub = X_scaled[sub_idx]

for idx, k in enumerate(K_values):
    labels_sub = kmeans_models[k].predict(X_sub)

    sil_avg = silhouette_score(X_sub, labels_sub)
    sample_silhouette_values = silhouette_samples(X_sub, labels_sub)

    ax = axes[idx]
    y_lower = 10

    for i in range(k):
        cluster_values = sample_silhouette_values[labels_sub == i]
        cluster_values.sort()

        size_cluster = len(cluster_values)
        y_upper = y_lower + size_cluster

        color = cm.nipy_spectral(float(i) / k)

        ax.fill_betweenx(
            np.arange(y_lower, y_upper),
            0,
            cluster_values,
            facecolor=color,
            edgecolor=color,
            alpha=0.7
        )

        ax.text(-0.05, y_lower + 0.5 * size_cluster, str(i))
        y_lower = y_upper + 10

    ax.axvline(
        x=sil_avg,
        color="red",
        linestyle="--",
        linewidth=1.5,
        label=f"Avg = {sil_avg:.3f}"
    )

    ax.set_title(f"Silhouette Plot for K = {k}")
    ax.set_xlabel("Silhouette Coefficient Values")
    ax.set_ylabel("Cluster Label")
    ax.set_xlim([-0.1, 1.0])
    ax.legend()
    ax.grid(True, alpha=0.3, linestyle="--")

plt.suptitle("Silhouette Analysis for K-Means Clustering")
plt.tight_layout()
plt.show()

# 2. Re-create the clustering evaluation results table from comparison_results list
clustering_results_df = pd.DataFrame(comparison_results)
results_table = clustering_results_df[
    clustering_results_df["K"].isin(K_values)
][["K", "WCSS", "Silhouette"]].copy()

results_table.columns = [
    "Number of Clusters (K)",
    "WCSS",
    "Silhouette Score"
]

print("\n=== K-MEANS EVALUATION RESULTS ===")
display(results_table.round(4))

# 3. Identify the K with the highest Silhouette Score
best_row_clust = clustering_results_df.loc[
    clustering_results_df["Silhouette"].idxmax()
]
best_k_clust = int(best_row_clust["K"])

print("\nBest K based on Silhouette Score:", best_k_clust)
print("Selected K for the project:", best_k_clust)


The K-Means algorithm was evaluated using three different numbers of clusters (K = 6, 7, and 8). The results showed that WCSS decreased as the number of clusters increased. The Silhouette Score also improved, reaching its highest value of 0.2011 at K = 8. Therefore, K = 8 was selected as the best option among the tested values. However, the relatively low Silhouette Scores indicate that the clusters have limited separation and some overlap.

Optimal K Selection

K = 8 was selected as the optimal value because it achieved the highest Silhouette Score (0.2011). Although K = 8 also had the lowest WCSS, its Silhouette Score was only slightly higher than K = 7, making K = 8 the better choice among the tested values.

## Clustering Final Conclusion

In [ ]:

conclusion_report = """
================================================================================
CLUSTERING (K-MEANS) - FINAL CONCLUSION
================================================================================

OBJECTIVE:
--------------------------------------------------

To discover natural patient groupings using K-Means clustering with:
• Three different K values (6, 7, 8) selected for evaluation
• Evaluation using Silhouette Coefficient and WCSS
• Visual interpretation of the clustering results
• Unsupervised learning (target/cardio variables excluded during clustering)


JUSTIFICATION FOR CHOOSING K=6, 7, 8:
--------------------------------------------------

Three values of K (6, 7, and 8) were selected to evaluate different
levels of patient grouping.

These values allow comparison of:
• Cluster compactness using WCSS
• Cluster separation using Silhouette Score
• The effect of increasing the number of clusters
• The balance between clustering quality and the number of groups

The final selection was based mainly on the Silhouette Score,
along with WCSS and the clustering visualizations.


EVALUATION METRICS COMPARISON:
--------------------------------------------------

K          Silhouette Score     WCSS (Inertia)       Interpretation
----------------------------------------------------------------------
6          0.1930               391,903.96           Initial grouping
7          0.2008               360,419.98           Improved separation
8          0.2011               344,744.00           Highest Silhouette Score

BEST K based on Silhouette Score: K=8 (Score: 0.2011)


CLUSTER CHARACTERISTICS:
--------------------------------------------------

For K=6:
   • Produces six groups of patients.
   • Achieves a Silhouette Score of 0.1930.
   • Has the highest WCSS among the tested values.

For K=7:
   • Produces seven groups of patients.
   • Improves the Silhouette Score to 0.2008.
   • Reduces WCSS compared to K=6, indicating more compact clusters.

For K=8:
   • Produces eight groups of patients.
   • Achieves the highest Silhouette Score (0.2011).
   • Has the lowest WCSS (344,743.9998) among the tested values.
   • Provides the best clustering result based on the evaluated metrics.


SILHOUETTE ANALYSIS INTERPRETATION:
--------------------------------------------------

Silhouette Score ranges from -1 to +1:
• +1.0  → Well-separated clusters
•  0.0  → Overlapping cluster boundaries
• -1.0  → Samples may be assigned to the wrong clusters

Our scores (0.1930 - 0.2011) indicate:
• Limited separation between the patient clusters.
• Some overlap between groups in the dataset.
• K=8 achieves the highest Silhouette Score (0.2011).
• The difference between K=7 and K=8 is very small.


RECOMMENDATIONS:
--------------------------------------------------

For the evaluated clustering results: Choose K=8
   • Achieves the highest Silhouette Score (0.2011).
   • Has the lowest WCSS among the tested values.
   • Provides the best result among K=6, 7, and 8 based on the selected metrics.

However, the small improvement over K=7 should be considered
when interpreting the final clustering results.


LIMITATIONS:
--------------------------------------------------

• Assumes spherical cluster geometry and may not capture complex
  relationships in the data.
• Sensitive to initial centroid initialization (mitigated using n_init=10).
• Requires feature scaling via StandardScaler (applied before clustering).
• Assigns hard cluster boundaries (each patient belongs to one cluster).
• Target labels (target/cardio) were excluded from clustering
  to maintain an unsupervised learning approach.
• The relatively low Silhouette Scores indicate limited cluster separation.


CONCLUSION:
--------------------------------------------------

K-Means clustering was applied to identify natural patient groupings
using scaled numerical features without relying on target labels.

The algorithm was evaluated using three different values of K:
6, 7, and 8. The results showed that WCSS decreased as the number
of clusters increased, while the Silhouette Score reached its
highest value of 0.2011 at K=8.

Therefore, K=8 was selected as the best option among the tested
values based on the evaluation metrics. However, the relatively
low Silhouette Scores indicate that the clusters have limited
separation and some overlap.

Overall, K=8 provided the best result among the evaluated
configurations, although further analysis of cluster characteristics
is needed to understand the patient groups.
"""

print(conclusion_report)


# Overall Findings and Discussion

The classification results showed that the Decision Tree using Entropy achieved the highest accuracy of 63.77% with 70% training and 30% testing, slightly outperforming Gini, which achieved 63.34%.

For clustering, K-Means was evaluated using different K values. K=8 achieved the highest Silhouette Score of 0.2011 among the tested values, making it the best choice.

###Classification VS Clustering

In [ ]:
import pandas as pd
from IPython.display import display

comparison_data = {
    "Aspect": [
        "Learning Type",
        "Target Variable",
        "Evaluation Metrics",
        "Goal",
        "Best Result",
        "Features Used",
        "Output"
    ],
    "Classification (Decision Tree)": [
        "Supervised",
        "cardio (0 or 1)",
        "Accuracy, Confusion Matrix",
        "Predict cardiovascular disease",
        "Entropy: 63.77% (70/30)",
        "Patient health features",
        "Disease / No Disease"
    ],
    "Clustering (K-Means)": [
        "Unsupervised",
        "None (Unsupervised)",
        "Silhouette Score, Elbow Method",
        "Group similar patients",
        "K=8, (Silhouette Score = 0.2011)",
        "Patient health features (excluding cardio)",
        "Eight Patient Clusters"
    ]
}

comparison_df = pd.DataFrame(comparison_data)

styled_table = (
    comparison_df.style
    .hide(axis="index")
    .set_properties(**{
        "border": "1.5px solid #333333",
        "padding": "12px",
        "text-align": "center",
        "white-space": "normal",
        "font-size": "13px"
    })
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("background-color", "#234E70"),
                ("color", "white"),
                ("border", "1.5px solid #333333"),
                ("padding", "12px"),
                ("text-align", "center")
            ]
        },
        {
            "selector": "table",
            "props": [
                ("border-collapse", "collapse"),
                ("width", "100%")
            ]
        }
    ])
    .apply(
        lambda row: [
            "background-color: #F1F5F9" if row.name % 2 else
            "background-color: #FFFFFF"
            for _ in row
        ],
        axis=1
    )
)

display(styled_table)


Classification is more suitable for the main objective of this project because it directly predicts whether a patient has cardiovascular disease using the known target variable (cardio). The Entropy Decision Tree achieved the highest accuracy of 63.77%.

In contrast, K-Means clustering identifies groups of patients with similar characteristics without using disease labels. The best result was K=8, with a Silhouette Score of 0.2011.

Therefore, classification is more suitable for disease prediction, while clustering provides additional information about similarities among patients. Both methods complement each other, but their evaluation scores cannot be compared directly because they measure different aspects of performance.

###Research Comparison

**Discussion Based on the Selected Research Papers**

The findings of our project can be compared with the two selected research papers that investigated cardiovascular disease prediction. Both papers highlight the importance of evaluating different approaches to improve prediction accuracy and better understand cardiovascular health data.

**Decision Tree Classification Analysis:**

In our project, we used Decision Tree classification with two splitting criteria, Entropy and Gini Index. We also tested three different training and testing splits to compare their performance and identify the most suitable configuration.

The best classification result was achieved using Entropy with a 70/30 training/testing split, reaching an accuracy of 63.77%. This indicates that the model was able to correctly classify approximately 64% of the test cases.

In comparison, Alqahtani et al. (2022) reported a higher accuracy of 88.70% in their study [1]. Similarly, Sadr et al. (2024) achieved an accuracy of 80.25% on the same cardiovascular dataset [2].

These results show that both research papers achieved better classification accuracy than our project. The differences may be related to data preprocessing, model configurations, and evaluation methods.

This comparison highlights the importance of testing different approaches and evaluating their results before selecting a suitable model for cardiovascular disease prediction.

**K-Means Clustering Analysis:**

In addition to classification, we applied K-Means clustering to identify groups of patients with similar health characteristics. Unlike classification, clustering does not use the known disease label to form groups.

We tested different values of K and evaluated the results using the Silhouette Score and Elbow Method to identify the most suitable number of clusters.

The best clustering result was obtained at K=8, with a Silhouette Score of 0.2011.

While the selected research papers mainly focused on cardiovascular disease prediction, our clustering analysis provided an additional way to explore patterns and similarities in the dataset. Therefore, a direct comparison of clustering scores with the papers was not possible.

**Overall Discussion:**

Our findings show that classification and clustering serve different purposes in cardiovascular data analysis. Decision Tree classification helps predict whether a patient has cardiovascular disease, while K-Means clustering helps identify similarities among patients.

Since our dataset includes a known target variable (cardio), classification is more suitable for the main objective of disease prediction. However, clustering is still useful for exploring the underlying structure of the data.

Overall, comparing our findings with the selected research papers shows the importance of evaluating model performance and understanding the strengths and limitations of different data mining techniques.



### Final Conclusion

In this project, we applied Decision Tree classification and K-Means clustering to the cardiovascular disease dataset. Decision Tree was used to predict whether a patient has cardiovascular disease, while K-Means was used to identify groups of patients with similar characteristics.

The best Decision Tree accuracy was 63.77% using Entropy with a 70/30 training/testing split. For K-Means clustering, the best result was K=8 with a Silhouette Score of 0.2011.

Overall, classification was more suitable for our project goal because the dataset contains a known target label (cardio). Clustering provided additional insights into similarities between patients.

###References

[1] A. Alqahtani, S. Alsubai, M. Sha, L. Vilcekova, and T. Javed, “Cardiovascular Disease Detection using Ensemble Learning,” Computational Intelligence and Neuroscience, vol. 2022, Art. no. 5267498, 2022. https://doi.org/10.1155/2022/5267498

[2] H. Sadr, A. Salari, M. T. Ashoobi, and M. Nazari, “Cardiovascular disease diagnosis: a holistic approach using the integration of machine learning and deep learning models,” European Journal of Medical Research, vol. 29, Art. no. 455, 2024. https://doi.org/10.1186/s40001-024-02044-7